# Model Training — Credit Risk XAI Dashboard

This notebook trains the Random Forest classifier used in the Credit Risk
XAI Dashboard for the bachelor thesis "Uncertainty Visualization in XAI
Dashboards: Effects on User Trust" (Schierbaum, 2026).

## Overview
- **Dataset:** German Credit Dataset (Hofmann, 1994), 1000 instances, 20 features
- **Model:** Random Forest Classifier, 100 trees, random_state=42
- **Train/Test Split:** 80/20, stratified by class label
- **Explainability:** SHAP TreeExplainer (Lundberg & Lee, 2017)
- **Output:** Trained model, SHAP explainer, and test data saved as .pkl files

## Pipeline
1. Load and inspect the German Credit Dataset
2. Encode categorical features using LabelEncoder
3. Split data into training (800) and test (200) sets
4. Train Random Forest classifier
5. Initialize SHAP TreeExplainer and verify output
6. Save all artifacts for use in the dashboard

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import shap

/opt/anaconda3/envs/xai-dashboard/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Dataset
The German Credit Dataset contains 1000 credit applications with 20 features
and a binary target variable (1 = good risk, 2 = bad risk). The target is
recoded to 0 = good, 1 = bad for consistency with scikit-learn conventions.

In [2]:
col_names = [
    "checking_account", "duration", "credit_history", "purpose",
    "credit_amount", "savings_account", "employment", "installment_rate",
    "personal_status", "other_debtors", "residence_since", "property",
    "age", "other_installments", "housing", "existing_credits",
    "job", "liable_people", "telephone", "foreign_worker", "target"
]

df = pd.read_csv("../data/german.data", sep=" ", header=None, names=col_names)
df["target"] = df["target"].map({1: 0, 2: 1})
print(df.shape)

(1000, 21)


## 3. Encode Categorical Features
Thirteen categorical features are encoded as integers using LabelEncoder.
Ordinal encoding is acceptable for tree-based models because decision tree
splits are threshold-based and do not assume metric relationships between
category codes.

In [3]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le
print("Enkodierung fertig ✓")

Enkodierung fertig ✓


/var/folders/sx/g2npqxpn749667fmdlv4xzj80000gn/T/ipykernel_22387/3985760223.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns.tolist()


## 4. Train/Test Split
The data is split 80/20 with stratification to preserve the class
distribution (700 good : 300 bad) across both subsets. A fixed random
seed ensures reproducibility.

In [4]:
X = df.drop(columns=["target"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training: {X_train.shape}, Test: {X_test.shape}")

Training: (800, 20), Test: (200, 20)


## 5. Train Random Forest Classifier
A Random Forest with 100 decision trees is trained on the training set.
The model achieves approximately 78% accuracy on the held-out test set.

In [5]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
acc = rf.score(X_test, y_test)
print(f"Accuracy: {acc:.3f}")

Accuracy: 0.780


## 6. Initialize SHAP TreeExplainer
The TreeExplainer computes exact Shapley values efficiently for
tree-based ensemble models (Lundberg & Lee, 2017). A quick validation
confirms correct output dimensions.

In [6]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test.iloc[:3])
print("SHAP OK — Shape:", shap_values.shape)

SHAP OK — Shape: (3, 20, 2)


## 7. Save Artifacts
All trained components are serialized using pickle for use in the
dashboard application. The saved files are:
- `rf_model.pkl` — trained Random Forest classifier
- `shap_explainer.pkl` — SHAP TreeExplainer instance
- `X_test.pkl` — test set features (200 × 20)
- `y_test.pkl` — test set labels
- `feature_names.pkl` — list of feature column names

In [7]:
with open("rf_model.pkl", "wb") as f:
    pickle.dump(rf, f)

with open("shap_explainer.pkl", "wb") as f:
    pickle.dump(explainer, f)

with open("X_test.pkl", "wb") as f:
    pickle.dump(X_test, f)

with open("y_test.pkl", "wb") as f:
    pickle.dump(y_test, f)

with open("feature_names.pkl", "wb") as f:
    pickle.dump(list(X.columns), f)

print("Alle Dateien gespeichert ✓")

Alle Dateien gespeichert ✓
